# Building a Multi-Agent Research Assistant with OpenAI Agents SDK and Olostep

This notebook builds a beginner-friendly multi-agent research assistant using:

- **OpenAI Agents SDK** for orchestration, specialist agents, tools, and structured outputs.
- **Olostep Python SDK** for Answer API, Search, Search with Scrape, and Scrape.

The final output is a **proper Markdown research report** for any user-provided research question, not just a short answer.


## Setup

Run this once. The `-U` matters because older `olostep` versions did not include `client.searches`.


In [ ]:
# %pip install -q -U openai-agents olostep python-dotenv pydantic

## Environment Variables

Create a `.env` file next to this notebook:

```bash
OPENAI_API_KEY=your_openai_api_key
OLOSTEP_API_KEY=your_olostep_api_key
```


In [2]:
import importlib.metadata
import json
import os
import textwrap
import warnings
from typing import Any

from dotenv import load_dotenv
from IPython.display import Markdown, display
from olostep import Olostep
from pydantic import BaseModel, Field

from agents import (
    Agent,
    Runner,
    custom_span,
    flush_traces,
    function_tool,
    gen_trace_id,
    trace,
)

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OLOSTEP_API_KEY = os.getenv("OLOSTEP_API_KEY")

## Olostep SDK Search With Scrape Demo

This standalone cell uses the exact SDK style requested.


In [3]:
client = Olostep(api_key=OLOSTEP_API_KEY)

search = client.searches.create(
    query="What are the most important recent developments in AI agents for business research?",
    limit=5,
    scrape_options={"formats": ["markdown"], "timeout": 25},
)

for link in search.links:
    print(link["url"], "-", len(link.get("markdown_content") or ""), "chars")


https://hbr.org/2025/11/the-ai-tools-that-are-transforming-market-research - 6658 chars
https://www.demandgenreport.com/industry-news/feature/ai-agents-revolutionize-b2b-marketing-in-2025-from-automation-to-strategy/51106/ - 66130 chars
https://mitsloan.mit.edu/ideas-made-to-matter/4-new-studies-about-agentic-ai-mit-initiative-digital-economy - 20014 chars
https://www.sciencedirect.com/science/article/abs/pii/S0148296325006228 - 60218 chars
https://www.bcg.com/capabilities/artificial-intelligence/ai-agents - 28638 chars


## Helpers

These helpers keep SDK responses compact and JSON-friendly for agent tool outputs.

Each Olostep helper also creates an OpenAI Agents trace span, so the trace shows which retrieval endpoint ran.


In [4]:
class OlostepError(RuntimeError):
    """Raised when an Olostep SDK request fails."""


def require_olostep_key() -> str:
    if not OLOSTEP_API_KEY:
        raise OlostepError(
            "OLOSTEP_API_KEY is not set. Add it to .env and rerun the setup cell."
        )
    return OLOSTEP_API_KEY


def get_olostep_client() -> Olostep:
    return Olostep(api_key=require_olostep_key())


def sdk_result_to_dict(result: Any) -> dict[str, Any]:
    if hasattr(result, "model_dump"):
        return result.model_dump()
    if hasattr(result, "__dict__"):
        return {
            key: value for key, value in vars(result).items() if not key.startswith("_")
        }
    return {"value": str(result)}


def compact_json(data: Any, max_chars: int = 8000) -> str:
    text = json.dumps(data, ensure_ascii=False, indent=2, default=str)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "\n... [truncated]"


def normalize_search_links(
    links: list[dict[str, Any]], limit: int = 8
) -> list[dict[str, Any]]:
    rows = []
    for link in links[:limit]:
        markdown = link.get("markdown_content") or ""
        rows.append(
            {
                "title": link.get("title") or "Untitled",
                "url": link.get("url") or "",
                "description": link.get("description") or "",
                "markdown_chars": len(markdown),
                "markdown_preview": markdown[:1500] if markdown else "",
            }
        )
    return rows


## Pydantic Structured Outputs

These models make the judge, source research, and final research report predictable.


In [5]:
class Judgment(BaseModel):
    is_good_enough: bool = Field(
        description="Whether the answer is sufficient for the user query."
    )
    score: float = Field(ge=0, le=1, description="Quality score from 0 to 1.")
    reason: str = Field(description="Short explanation of the decision.")
    missing_information: list[str] = Field(
        default_factory=list, description="Important gaps to fix."
    )


class SourceResearchReport(BaseModel):
    key_findings: list[str] = Field(
        description="Concise findings from gathered sources."
    )
    important_urls: list[str] = Field(
        description="Only the most important URLs used for synthesis."
    )
    source_notes: list[str] = Field(
        description="Brief notes connecting sources to findings."
    )
    remaining_gaps: list[str] = Field(
        default_factory=list, description="Gaps that could not be resolved."
    )


class MarkdownResearchReport(BaseModel):
    title: str = Field(description="Research report title.")
    executive_summary: str = Field(description="Short answer-first summary.")
    key_findings: list[str] = Field(description="Most important findings.")
    markdown_report: str = Field(
        description="Complete Markdown report with polished headings, clear analysis, reader-friendly structure, and citations."
    )
    citations: list[str] = Field(
        default_factory=list, description="Source URLs used in the report."
    )
    confidence: str = Field(description="Low, medium, or high confidence.")
    method_used: str = Field(description="Retrieval path used by the manager agent.")


## Olostep Tool Functions

These are the four requested functions. Each one is exposed as an OpenAI Agents SDK `function_tool`.


In [6]:
@function_tool
async def answer_query(query: str) -> str:
    """Answer a natural-language research query using Olostep Answer API."""
    try:
        with custom_span("olostep.answer_query", {"query": query}):
            result = get_olostep_client().answers.create(task=query)
            return compact_json(sdk_result_to_dict(result))

    except Exception as exc:
        raise OlostepError(f"Olostep Answer API failed: {exc}") from exc


@function_tool
async def search_web(query: str, limit: int = 8) -> str:
    """Search the web using Olostep Search and return normalized results."""
    try:
        with custom_span("olostep.search_web", {"query": query, "limit": limit}):
            search = get_olostep_client().searches.create(
                query=query,
                limit=limit,
            )

            data = sdk_result_to_dict(search)

            return compact_json(
                {
                    "query": query,
                    "results": normalize_search_links(
                        data.get("links", []),
                        limit=limit,
                    ),
                    "raw": data,
                }
            )

    except Exception as exc:
        raise OlostepError(f"Olostep Search API failed: {exc}") from exc


@function_tool
async def search_with_scrape(query: str, limit: int = 5) -> str:
    """Search the web and scrape each returned link using Olostep Search with Scrape."""
    scrape_options = {
        "formats": ["markdown"],
        "timeout": 25,
    }

    try:
        with custom_span(
            "olostep.search_with_scrape",
            {
                "query": query,
                "limit": limit,
                "scrape_options": scrape_options,
            },
        ):
            search = get_olostep_client().searches.create(
                query=query,
                limit=limit,
                scrape_options=scrape_options,
            )

            data = sdk_result_to_dict(search)

            return compact_json(
                {
                    "query": query,
                    "results": normalize_search_links(
                        data.get("links", []),
                        limit=limit,
                    ),
                    "raw": data,
                },
                max_chars=12000,
            )

    except Exception as exc:
        raise OlostepError(f"Olostep Search with Scrape failed: {exc}") from exc


@function_tool
async def scrape_url(url: str) -> str:
    """Scrape one URL with Olostep and return compact page content."""
    try:
        with custom_span(
            "olostep.scrape_url",
            {
                "url": url,
                "formats": ["markdown"],
            },
        ):
            scrape = get_olostep_client().scrapes.create(
                url=url,
                formats=["markdown"],
            )

            return compact_json(
                {
                    "url": url,
                    "scrape": sdk_result_to_dict(scrape),
                },
                max_chars=10000,
            )

    except Exception as exc:
        raise OlostepError(f"Olostep Scrape API failed: {exc}") from exc

## Specialist Agents

The manager will call these agents as tools.

This is the orchestration pattern: specialized agents remain separate, but the manager controls when to invoke them.


In [7]:
MODEL = "gpt-5.4-mini"

judge_agent = Agent(
    name="Judge agent",
    model=MODEL,
    instructions=(
        "You judge whether the provided answer is good enough for the original research question. "
        "Reward direct, specific, source-backed answers. Reject vague, stale, or unsupported answers. "
        "Return only the structured judgment."
    ),
    output_type=Judgment,
)

source_research_agent = Agent(
    name="Source research agent",
    model=MODEL,
    instructions=(
        "You gather evidence for a research report using only the provided Olostep tools. "
        "First try search_with_scrape for the original query. If the scraped search result is weak, "
        "run two or three targeted search_web calls, select only the most important URLs, scrape those URLs, "
        "and summarize the evidence. Prefer official, primary, reputable, and recent sources. "
        "Return only the structured source research report."
    ),
    tools=[search_web, search_with_scrape, scrape_url],
    output_type=SourceResearchReport,
)

analyst_agent = Agent(
    name="Analyst agent",
    model=MODEL,
    instructions=(
        "You write a proper Markdown research report from the evidence. "
        "Write for a professional reader who wants a clear, polished research brief on any topic. "
        "Adapt the report to the user's question. The markdown_report must be substantial, easy to scan, and use these general sections only: "
        "Executive Summary, Key Findings, Context, Evidence Review, Detailed Analysis, Implications, Source Notes, and References. "
        "If the topic is event-driven, include timeline details inside Context or Detailed Analysis instead of adding a separate Timeline section. "
        "If the topic is comparative, include a compact comparison table inside Detailed Analysis. "
        "Do not include sections titled Limitations, Next Steps, Recommendations, or Action Items. "
        "Avoid bare caveats like 'I relied on...'. Instead, integrate source quality naturally in Source Notes. "
        "Use short paragraphs, bullets where helpful, and citations as Markdown links or URL bullets. "
        "Add enough context that a non-expert reader understands the issue, why it matters, and what evidence supports it. "
        "Return only the structured report."
    ),
    output_type=MarkdownResearchReport,
)


## Manager Agent As Orchestrator

Here the manager is given the specialist agents as tools. The notebook will run only this manager agent for the real research workflow.


In [8]:
judge_tool = judge_agent.as_tool(
    tool_name="judge_answer_quality",
    tool_description="Judge whether an answer or evidence is good enough for the original research question.",
)

source_research_tool = source_research_agent.as_tool(
    tool_name="run_source_research",
    tool_description="Run Olostep search-with-scrape, targeted searches, URL selection, and URL scraping to gather source evidence.",
)

analyst_tool = analyst_agent.as_tool(
    tool_name="write_markdown_research_report",
    tool_description="Write the final structured Markdown research report from the gathered evidence.",
)

manager_agent = Agent(
    name="Manager research agent",
    model=MODEL,
    instructions=(
        "You are the orchestrator for a multi-agent research assistant. Follow this exact policy:\n"
        "1. Always call answer_query first for the user's question.\n"
        "2. Call judge_answer_quality on that Answer API result.\n"
        "3. If the judge says the answer is good enough, call write_markdown_research_report using the answer result.\n"
        "4. If the judge says the answer is not good enough, call run_source_research. The source researcher must use search_with_scrape first, then targeted searches and scrape_url if needed.\n"
        "5. Call write_markdown_research_report using all evidence.\n"
        "6. Return a complete MarkdownResearchReport. Do not return a casual chat answer."
    ),
    tools=[answer_query, judge_tool, source_research_tool, analyst_tool],
    output_type=MarkdownResearchReport,
)


## Run The Orchestrated Research Assistant With Tracing

This is now a single top-level manager run. The manager decides which tools and specialist agents to call.


In [9]:
def openai_trace_url(trace_id: str) -> str:
    return f"https://platform.openai.com/logs/trace?trace_id={trace_id}"


async def run_research_assistant(query: str) -> MarkdownResearchReport:
    if not OPENAI_API_KEY:
        raise RuntimeError(
            "OPENAI_API_KEY is not set. Add it to .env and rerun the setup cell."
        )
    require_olostep_key()

    trace_id = gen_trace_id()
    print("OpenAI trace ID:", trace_id)
    print("OpenAI trace URL:", openai_trace_url(trace_id))

    prompt = f"""
Research question:
{query}

Return a polished, reader-friendly Markdown research report with substantial detail for the user's specific question. Follow the required workflow exactly:
- Answer API first.
- Judge agent second.
- If weak, source research agent with search_with_scrape, targeted search_web calls, URL selection, and scrape_url.
- Analyst agent writes the final Markdown report. Do not include Limitations or Next Steps sections.
"""
    with trace(
        workflow_name="multi_agent_research_assistant_olostep",
        trace_id=trace_id,
        metadata={
            "query": query,
            "notebook": "multi_agent_research_assistant_openai_agents_olostep",
        },
    ):
        with custom_span("manager.run", {"query": query}):
            result = await Runner.run(manager_agent, prompt, max_turns=20)

    flush_traces()
    print(
        "Trace flushed. Open the URL above to inspect manager, specialist agents, tools, and Olostep spans."
    )
    return result.final_output


## Example Query

In [10]:
example_query = "What's going on with OpenAI's Sora shutting down?"

report = await run_research_assistant(example_query)
display(Markdown(report.markdown_report))


OpenAI trace ID: trace_ec98c6714b194dfca0ee61d016c3c533
OpenAI trace URL: https://platform.openai.com/logs/trace?trace_id=trace_ec98c6714b194dfca0ee61d016c3c533


C:\Users\abida\AppData\Roaming\Python\Python313\site-packages\olostep\backend\caller.py:491: UserWarning: The API response from 'Create Scrape' contains 1 extra field(s) not defined in the SDK model: cost_usd. This may indicate new API features. Please check for a newer SDK version on Slack (best place to ask), PyPI or Github. (current: 1.0.4). Visit https://docs.olostep.com/sdks/python for updates.
  return await self._invoke(


Trace flushed. Open the URL above to inspect manager, specialist agents, tools, and Olostep spans.


## Executive Summary

OpenAI’s Sora is **not** being shut down in a single instant. It is being **phased out**. OpenAI’s Help Center says the Sora web and app experiences were discontinued on **April 26, 2026**, while the **Sora API** will be discontinued on **September 24, 2026**. Reporting from [The New York Times](https://www.nytimes.com/2026/03/24/technology/openai-shutting-down-sora.html) and [CNN](https://www.cnn.com/2026/03/24/tech/openai-sora-video-app-shutting-down) says the move affects the consumer app and a broader video-generation/professional service tied to Sora.

## Key Findings

- OpenAI describes the change as a **discontinuation in phases**, not an immediate full shutdown.
- The **Sora web and app experiences** were discontinued on **April 26, 2026**.
- The **Sora API** remains available until **September 24, 2026**.
- The **consumer app/web experience** is the first major product being ended.
- Reporting indicates the wind-down also reaches a **professional video-generation service** tied to Sora.
- OpenAI’s stated rationale includes a shift toward **world-simulation research and robotics** and the **high compute costs** of operating the service.
- OpenAI is reportedly exploring **export/preservation** options for user content as part of the transition.

## What’s Actually Happening

Sora is OpenAI’s video-generation product ecosystem, spanning consumer-facing web/app experiences and an API used for programmatic access. The current situation is best understood as a **managed product retirement**, not a service outage. That distinction matters: an outage is accidental and temporary, while a discontinuation is planned and time-bound.

The wind-down appears to be part of a broader strategic reallocation. The *New York Times* says the shutdown affects both the consumer app and a professional internet service tied to Sora, and connects the move to broader changes after the Disney licensing deal. CNN adds that OpenAI is emphasizing world-simulation research and robotics, suggesting Sora’s current product form is being deprioritized in favor of other priorities.

## What Is Being Shut Down

OpenAI is ending multiple parts of the Sora product stack:

1. **Sora web and app experiences** — discontinued first, on April 26, 2026.
2. **Sora API** — remains available temporarily and is scheduled to end on September 24, 2026.
3. **Related consumer/professional video-generation service** — reported by the *New York Times* and CNN as part of the broader wind-down.

## Why It Is Happening

The evidence points to two overlapping drivers:

- **Strategic reprioritization**: OpenAI is shifting attention toward **world-simulation research and robotics**, according to CNN.
- **Operational economics**: CNN reports the service carried **high compute costs**, forcing trade-offs in how OpenAI allocates resources.

The *New York Times* adds that the move follows broader strategic changes after the **Disney licensing deal**, suggesting Sora’s role may have changed as OpenAI’s product and partnership strategy evolved.

## Timeline

| Component | Status | Date | Notes |
|---|---:|---:|---|
| Sora web/app experience | Discontinued | April 26, 2026 | Main consumer-facing product ended first |
| Sora API | Scheduled discontinuation | September 24, 2026 | Programmatic access remains available until later cutoff |
| Related professional video service | Wound down | Reported in March 2026 coverage | Described in news reporting as tied to Sora |

## What This Means for Users

For users, the key takeaway is that Sora’s shutdown is **not a sudden outage** and not a single simultaneous removal of every access path. Instead, OpenAI is phasing out the consumer experience first and the API later, which gives users more time to adapt.

For the broader market, the move signals that even prominent AI media products can be constrained by **compute economics** and shifting research priorities. It also suggests OpenAI is choosing to concentrate resources on longer-horizon capabilities—especially world simulation and robotics—rather than continue supporting the current Sora product structure indefinitely.

## Bottom Line

Sora is being **deliberately discontinued in stages**, not experiencing a technical failure. The consumer web/app product ended first, the API is next, and the rationale appears to be a mix of strategic refocus and the high cost of running the service.

## Sources

- [OpenAI Help Center: What to know about the Sora discontinuation](https://help.openai.com/en/articles/20001152-what-to-know-about-the-sora-discontinuation)
- [The New York Times: OpenAI Shutting Down Sora](https://www.nytimes.com/2026/03/24/technology/openai-shutting-down-sora.html)
- [CNN: OpenAI Sora video app shutting down](https://www.cnn.com/2026/03/24/tech/openai-sora-video-app-shutting-down)

![image_1.png](image_1.png)

## Save A Markdown Report

After running the live example, you can save the report to a `.md` file.


In [11]:
# After running the live example above:
output_path = "research_report.md"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(report.markdown_report)
print(f"Saved {output_path}")


Saved research_report.md
